# Seattle Fault Line Data Exploration
## Initial EDA and Feature Analysis

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

# Load seismic data
df = pd.read_csv('../data/raw/seismic_events_2018_2024.csv')
print(f"Dataset shape: {df.shape}")
print(f"\nFirst few rows:")
print(df.head())

In [ ]:
# Basic statistics
print("Magnitude Statistics:")
print(df['magnitude'].describe())
print("\nDepth Statistics (km):")
print(df['depth_km'].describe())
print("\nData Types:")
print(df.dtypes)

In [ ]:
# Visualization: Magnitude distribution
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Histogram of magnitudes
axes[0, 0].hist(df['magnitude'], bins=30, color='steelblue', edgecolor='black')
axes[0, 0].set_xlabel('Magnitude')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].set_title('Distribution of Earthquake Magnitudes')

# Depth vs Magnitude scatter
axes[0, 1].scatter(df['magnitude'], df['depth_km'], alpha=0.5, c=df['event_id'], cmap='viridis')
axes[0, 1].set_xlabel('Magnitude')
axes[0, 1].set_ylabel('Depth (km)')
axes[0, 1].set_title('Magnitude vs Depth')

# Latitude vs Longitude (epicenter map)
scatter = axes[1, 0].scatter(df['longitude'], df['latitude'], 
                              c=df['magnitude'], s=df['magnitude']*50, 
                              alpha=0.6, cmap='hot')
axes[1, 0].set_xlabel('Longitude')
axes[1, 0].set_ylabel('Latitude')
axes[1, 0].set_title('Epicenter Distribution (Size = Magnitude)')
plt.colorbar(scatter, ax=axes[1, 0], label='Magnitude')

# Energy release over time
df['timestamp'] = pd.to_datetime(df['timestamp'])
df_sorted = df.sort_values('timestamp')
axes[1, 1].plot(df_sorted['timestamp'], df_sorted['energy_joules'], marker='o', linestyle='-', markersize=3)
axes[1, 1].set_xlabel('Time')
axes[1, 1].set_ylabel('Energy Released (Joules)')
axes[1, 1].set_title('Energy Release Timeline')
axes[1, 1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('../figures/01_data_exploration.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Feature correlation analysis
correlation_cols = ['magnitude', 'depth_km', 'energy_joules', 'p_wave_velocity', 's_wave_velocity']
correlation_matrix = df[correlation_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, 
            square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.savefig('../figures/correlation_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Identify potential fault zones
# Using clustering on geographic coordinates
from sklearn.cluster import DBSCAN

coords = df[['latitude', 'longitude', 'depth_km']].values
clustering = DBSCAN(eps=0.15, min_samples=5).fit(coords)
df['fault_zone_cluster'] = clustering.labels_

print(f"Number of identified fault zones: {len(set(df['fault_zone_cluster'])) - 1}")
print(f"\nEvents per fault zone:")
print(df['fault_zone_cluster'].value_counts().head(10))